# 3D 卷积详解：原理、实现与视频 / 医学影像应用

> 本 Notebook 系统介绍三维卷积（3D Convolution）的核心原理，从 2D 卷积扩展到 3D 卷积的思路出发，详细讲解 `nn.Conv3d`、3D 池化、3D 转置卷积等基础算子，并通过合成数据演示在视频理解（时序建模）和医学影像（CT/MRI 体素分析）两大场景中的完整应用，最后对比 3D 卷积的几种高效变体（(2+1)D 卷积、伪 3D 卷积）的适用场景。

## 内容概览

| 章节 | 内容 |
|------|------|
| 一、环境配置 | 导入库、检测设备 |
| 二、从 2D 到 3D | 维度扩展的直觉与张量形状 |
| 三、nn.Conv3d 详解 | 参数说明、输出尺寸公式、权重结构 |
| 四、3D 池化 | MaxPool3d、AvgPool3d、AdaptiveAvgPool3d |
| 五、3D 转置卷积 | ConvTranspose3d 上采样原理与实现 |
| 六、3D 卷积网络实战 | 简化版 C3D 视频分类网络 |
| 七、医学影像应用 | 3D U-Net 编解码器模块 |
| 八、高效变体对比 | (2+1)D 卷积、Pseudo-3D 及场景对比 |
| 九、总结 | 各方法适用场景速查表 |

## 一、环境配置与依赖导入

In [ ]:
import matplotlib as mpl                    # matplotlib：Python 最常用的绘图库
import matplotlib.pyplot as plt             # pyplot 子模块：提供类似 MATLAB 的绘图接口
%matplotlib inline
# Jupyter Magic 命令：让图像直接在 notebook 中内嵌显示

import numpy as np                          # NumPy：高性能数值计算库，提供多维数组 ndarray
import sys                                  # sys：Python 解释器相关信息
from mpl_toolkits.mplot3d import Axes3D     # mpl_toolkits：matplotlib 3D 绘图工具包

import torch                                # PyTorch 主库
import torch.nn as nn                       # nn 模块：封装各种神经网络层
import torch.nn.functional as F             # F 模块：以函数形式提供激活、池化等操作

print(sys.version_info)                     # 打印 Python 解释器版本信息
for module in mpl, np, torch:               # 遍历主要库，打印名称与版本号
    print(module.__name__, module.__version__)  # 格式：库名 版本号

# 选择运算设备
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")  # 有 GPU 用 cuda:0，否则用 CPU
print("使用设备：", device)  # 打印当前所用设备

# 设置 matplotlib 中文字体，避免中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']  # 中文字体优先级列表
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

seed = 42                      # 随机种子，保证实验可复现
torch.manual_seed(seed)        # 设置 PyTorch CPU 随机种子
np.random.seed(seed)           # 设置 NumPy 随机种子

## 二、从 2D 卷积到 3D 卷积

### 2.1 维度扩展的直觉

| 维度 | 输入张量形状 | 卷积核形状 | 典型应用 |
|------|------------|----------|--------|
| **1D 卷积** | `(N, C, L)` | `(C_out, C_in, k)` | 音频、时间序列、文本 |
| **2D 卷积** | `(N, C, H, W)` | `(C_out, C_in, kH, kW)` | 图像分类、目标检测、分割 |
| **3D 卷积** | `(N, C, D, H, W)` | `(C_out, C_in, kD, kH, kW)` | 视频理解、CT/MRI 医学影像 |

**2D 卷积**在图像的 **高（H）× 宽（W）** 两个空间维度上滑动；  
**3D 卷积**额外增加了一个 **深度（D）** 维度，可以是：
- **时间维度**（视频中的帧序列）：捕获**时序运动信息**
- **空间深度维度**（CT/MRI 中的切片序列）：捕获**立体体素结构**

### 2.2 张量形状对比可视化

In [ ]:
# ── 构造 2D 与 3D 输入张量，直观感受形状差异 ────────────────────────────────────

torch.manual_seed(seed)  # 固定随机种子

# 2D 输入：模拟一张 RGB 图像，batch=1
# 形状含义：(batch_size, channels, height, width)
img_2d = torch.randn(1, 3, 32, 32)  # (N=1, C=3, H=32, W=32)，dtype=float32

# 3D 输入（视频）：模拟 8 帧的 RGB 视频片段，batch=1
# 形状含义：(batch_size, channels, depth/frames, height, width)
video_3d = torch.randn(1, 3, 8, 32, 32)  # (N=1, C=3, D=8帧, H=32, W=32)，dtype=float32

# 3D 输入（医学影像）：模拟 CT 扫描体数据，batch=1，单通道
# 形状含义：(batch_size, channels, depth/slices, height, width)
ct_3d = torch.randn(1, 1, 64, 64, 64)  # (N=1, C=1, D=64切片, H=64, W=64)，dtype=float32

print("形状对比：")
print(f"  2D 图像张量:      {img_2d.shape}     → (N, C, H, W)")           # 4 维
print(f"  3D 视频张量:    {video_3d.shape}  → (N, C, D, H, W)")          # 5 维
print(f"  3D CT 体数据:  {ct_3d.shape}  → (N, C, D, H, W)")
print()
print("内存占用（float32，字节数）：")
print(f"  2D 图像:    {img_2d.numel() * 4:>10,} bytes")    # numel()：元素总数；×4 bytes/float32
print(f"  3D 视频:    {video_3d.numel() * 4:>10,} bytes")   # 3D 视频内存为 2D 图像的 D 倍
print(f"  3D CT:      {ct_3d.numel() * 4:>10,} bytes")      # CT 体数据内存显著更大

# ── 3D 输入结构示意图（文本可视化）────────────────────────────────────────────
print()
print("3D 卷积输入结构示意（视频场景）：")
print("  帧 t=0  帧 t=1  帧 t=2  ...  帧 t=7")
print("  ┌────┐  ┌────┐  ┌────┐       ┌────┐")
print("  │ H  │  │    │  │    │  ...  │    │  ← 每帧 H×W 的 RGB 图像")
print("  │  W │  │    │  │    │       │    │")
print("  └────┘  └────┘  └────┘       └────┘")
print("  ├──────────── D（时间维）─────────────┤")
print("  3D 卷积核同时在 D/H/W 三个维度滑动，捕获时空联合特征")

### 2.3 2D 卷积与 3D 卷积的核心区别

In [ ]:
# ── 用简单示例对比 2D 与 3D 卷积的参数量差异 ──────────────────────────────────

torch.manual_seed(seed)  # 固定初始化

# 2D 卷积层：处理图像（H×W 空间）
conv2d_layer = nn.Conv2d(
    in_channels=3,      # 输入通道：RGB 3 通道
    out_channels=64,    # 输出通道：64 个卷积核
    kernel_size=3,      # 2D 卷积核大小：3×3
    padding=1,          # same padding：保持 H、W 不变
    bias=False          # 不使用偏置
)  # 参数量 = 64 × 3 × 3 × 3 = 1728

# 3D 卷积层：处理视频/体数据（D×H×W 时空）
conv3d_layer = nn.Conv3d(
    in_channels=3,      # 输入通道：RGB 3 通道
    out_channels=64,    # 输出通道：64 个 3D 卷积核
    kernel_size=3,      # 3D 卷积核大小：3×3×3（D×H×W 方向各 3）
    padding=1,          # same padding：保持 D、H、W 不变
    bias=False          # 不使用偏置
)  # 参数量 = 64 × 3 × 3 × 3 × 3 = 5184（是 2D 的 3 倍）

# 计算参数量：遍历所有参数张量，累加元素数目
params_2d = sum(p.numel() for p in conv2d_layer.parameters())  # Conv2d 总参数数（int）
params_3d = sum(p.numel() for p in conv3d_layer.parameters())  # Conv3d 总参数数（int）

print("Conv2d vs Conv3d 参数对比（out=64, in=3, kernel=3）：")
print(f"  Conv2d 参数量: {params_2d:,}  （kernel: 3×3×3×64 = {3*3*3*64}）")    # 1728
print(f"  Conv3d 参数量: {params_3d:,}  （kernel: 3×3×3×3×64 = {3*3*3*3*64}）") # 5184
print(f"  参数量之比:    {params_3d / params_2d:.1f}×（3D 卷积参数量是 2D 的 kernel_d 倍）")
print()

# 前向传播验证形状
with torch.no_grad():  # 关闭梯度追踪（仅做推理演示）
    out_2d = conv2d_layer(img_2d)      # 2D 前向：(1,3,32,32) → (1,64,32,32)
    out_3d = conv3d_layer(video_3d)    # 3D 前向：(1,3,8,32,32) → (1,64,8,32,32)

print("前向传播形状验证：")
print(f"  Conv2d: {img_2d.shape} → {out_2d.shape}")
print(f"  Conv3d: {video_3d.shape} → {out_3d.shape}")

## 三、`nn.Conv3d` 详解

### 3.1 接口与参数说明

```python
nn.Conv3d(
    in_channels,                  # 输入通道数 C_in
    out_channels,                 # 输出通道数 C_out（卷积核数量）
    kernel_size,                  # 卷积核尺寸：int 或 (kD, kH, kW) 元组
    stride      = 1,              # 步长：int 或 (sD, sH, sW)，默认 1
    padding     = 0,              # 填充：int 或 (pD, pH, pW)，默认 0
    dilation    = 1,              # 空洞率：int 或 (dD, dH, dW)，默认 1
    groups      = 1,              # 分组卷积数，默认 1（标准卷积）
    bias        = True,           # 是否使用偏置项，默认 True
    padding_mode = 'zeros'        # 填充模式：'zeros'/'reflect'/'replicate'/'circular'
)
```

### 3.2 输出尺寸公式

对于 D、H、W 三个维度，输出尺寸的计算公式相同：

$$
D_{\text{out}} = \left\lfloor \frac{D_{\text{in}} + 2p_D - d_D(k_D - 1) - 1}{s_D} + 1 \right\rfloor
$$

其中 $p$ = padding，$d$ = dilation，$k$ = kernel_size，$s$ = stride。

**Same Padding 条件**（输出与输入尺寸相等，stride=1，dilation=1）：`padding = kernel_size // 2`

In [ ]:
def calc_conv3d_out(d_in, h_in, w_in, kernel_size, stride=1, padding=0, dilation=1):
    """计算 Conv3d 的输出尺寸。

    参数:
        d_in, h_in, w_in (int): 输入的深度、高度、宽度
        kernel_size (int 或 tuple): 卷积核尺寸，若为 int 则三维相同
        stride (int 或 tuple): 步长，默认 1
        padding (int 或 tuple): 填充，默认 0
        dilation (int 或 tuple): 空洞率，默认 1

    返回:
        tuple: (D_out, H_out, W_out) 输出的深度、高度、宽度（int）
    """
    # 将 int 参数统一转换为 3 元素元组，方便对三个维度分别计算
    def to3(v):  # 辅助函数：将 int 转为 (v, v, v) 元组
        return (v, v, v) if isinstance(v, int) else v  # isinstance：检查 v 是否为 int 类型

    k = to3(kernel_size)  # 卷积核三维尺寸元组 (kD, kH, kW)
    s = to3(stride)       # 步长三维元组 (sD, sH, sW)
    p = to3(padding)      # 填充三维元组 (pD, pH, pW)
    d = to3(dilation)     # 空洞率三维元组 (dD, dH, dW)

    # 套用输出尺寸公式：floor((in + 2p - d*(k-1) - 1) / s + 1)
    d_out = (d_in + 2*p[0] - d[0]*(k[0]-1) - 1) // s[0] + 1  # D 维输出尺寸（int）
    h_out = (h_in + 2*p[1] - d[1]*(k[1]-1) - 1) // s[1] + 1  # H 维输出尺寸（int）
    w_out = (w_in + 2*p[2] - d[2]*(k[2]-1) - 1) // s[2] + 1  # W 维输出尺寸（int）

    return d_out, h_out, w_out  # 返回三元组 (D_out, H_out, W_out)


# ── 演示几种常见配置的输出尺寸 ───────────────────────────────────────────────
print("Conv3d 输出尺寸计算演示（输入: D=8, H=32, W=32）：")
print()

# 配置 1：same padding（stride=1，输出与输入等尺寸）
d1, h1, w1 = calc_conv3d_out(8, 32, 32, kernel_size=3, stride=1, padding=1)  # same: pad=kernel//2
print(f"  kernel=3, stride=1, padding=1 (same):  输出 ({d1}, {h1}, {w1})")  # (8, 32, 32)

# 配置 2：stride=2（空间下采样，常用于特征提取）
d2, h2, w2 = calc_conv3d_out(8, 32, 32, kernel_size=3, stride=2, padding=1)  # 步长2：尺寸约减半
print(f"  kernel=3, stride=2, padding=1:         输出 ({d2}, {h2}, {w2})")  # (4, 16, 16)

# 配置 3：各维度不同步长（时间维不下采样，空间维下采样）
d3, h3, w3 = calc_conv3d_out(8, 32, 32, kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1))  # 仅空间下采样
print(f"  kernel=(1,3,3), stride=(1,2,2):        输出 ({d3}, {h3}, {w3})")  # (8, 16, 16)

# 配置 4：空洞卷积（dilation=2，扩大感受野）
d4, h4, w4 = calc_conv3d_out(8, 32, 32, kernel_size=3, stride=1, padding=2, dilation=2)  # 空洞率 2
print(f"  kernel=3, stride=1, dilation=2, pad=2: 输出 ({d4}, {h4}, {w4})")  # (8, 32, 32)

print()
print("提示：设置 padding=kernel_size//2（stride=1，dilation=1）可获得 same padding")

### 3.3 Conv3d 权重结构与感受野

In [ ]:
torch.manual_seed(seed)  # 固定权重初始化

# 构建一个单卷积核的 Conv3d 层，便于检查权重结构
single_conv3d = nn.Conv3d(
    in_channels=1,   # 输入通道：1（灰度/单通道）
    out_channels=1,  # 输出通道：1（单个卷积核）
    kernel_size=3,   # 3×3×3 卷积核
    padding=1,       # same padding
    bias=False       # 关闭偏置，只关注卷积核权重
)  # 权重张量形状：(out_channels, in_channels, kD, kH, kW) = (1, 1, 3, 3, 3)

weight = single_conv3d.weight  # 获取权重张量，形状 (1, 1, 3, 3, 3)
print("Conv3d 权重张量形状（out_c, in_c, kD, kH, kW）:", weight.shape)
print("总权重参数数量:", weight.numel(), "= 1×1×3×3×3")
print()
# 按深度维度（kD）切片打印每一层的 3×3 权重，展示 3D 卷积核的立体结构
for d_idx in range(weight.shape[2]):  # weight.shape[2] = kD = 3
    # weight[0, 0, d_idx]：第 0 个输出核、第 0 个输入通道、第 d_idx 深度切片，形状 (3, 3)
    print(f"  卷积核 深度切片 d={d_idx}（3×3）：")
    print(weight.data[0, 0, d_idx].numpy().round(3))  # .data 去掉梯度，.numpy() 转为数组
    print()

# ── 可视化 3D 卷积核（热力图方式展示三个深度切片）────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # 创建 1×3 子图布局（3 个深度切片）
k_data = weight.data[0, 0].numpy()              # 取出 (kD=3, kH=3, kW=3) 的 numpy 数组

for i, ax in enumerate(axes):  # 遍历 3 个深度切片（d=0,1,2）
    im = ax.imshow(k_data[i], cmap='RdBu_r', vmin=k_data.min(), vmax=k_data.max())  # 热力图
    ax.set_title(f'卷积核切片 d={i}', fontsize=11)  # 子图标题
    ax.set_xticks(range(3))  # x 轴刻度：0,1,2 对应 kW
    ax.set_yticks(range(3))  # y 轴刻度：0,1,2 对应 kH
    # 在每个格子中标注权重数值
    for row in range(3):          # 遍历 kH 维度
        for col in range(3):      # 遍历 kW 维度
            ax.text(col, row, f'{k_data[i, row, col]:.2f}',  # 标注权重值（保留 2 位小数）
                    ha='center', va='center', fontsize=8, color='black')  # 居中对齐

fig.colorbar(im, ax=axes.tolist(), shrink=0.8)  # 添加共享颜色条
plt.suptitle('3D 卷积核（1×1×3×3×3）的三个深度切片', fontsize=13)  # 整体标题
plt.tight_layout()  # 自动调整间距
plt.show()          # 渲染显示

## 四、3D 池化

3D 池化是对 `(D, H, W)` 三维空间同时做聚合操作，用于缩小特征图尺寸、增大感受野。

| 池化类型 | PyTorch 类 | 特点 |
|---------|-----------|------|
| 最大池化 | `nn.MaxPool3d` | 取池化窗口内最大值；保留最显著特征 |
| 平均池化 | `nn.AvgPool3d` | 取窗口内均值；输出更平滑 |
| 自适应平均池化 | `nn.AdaptiveAvgPool3d` | 指定输出尺寸，自动计算步长；常用于网络末尾全局池化 |

In [ ]:
torch.manual_seed(seed)  # 固定随机种子

feat_3d = torch.randn(1, 64, 8, 32, 32)  # 模拟 3D 特征图，形状 (N=1, C=64, D=8, H=32, W=32)

# ── MaxPool3d：最大池化 ──────────────────────────────────────────────────────
max_pool = nn.MaxPool3d(
    kernel_size=2,  # 池化窗口 2×2×2（D/H/W 各取 2）
    stride=2        # 步长 2：每个维度下采样 2 倍
)  # 输出尺寸：D/2 × H/2 × W/2

# ── AvgPool3d：平均池化 ──────────────────────────────────────────────────────
avg_pool = nn.AvgPool3d(
    kernel_size=2,  # 池化窗口 2×2×2
    stride=2        # 步长 2
)  # 输出尺寸同 MaxPool3d

# ── AdaptiveAvgPool3d：自适应平均池化 ─────────────────────────────────────────
# 直接指定输出空间尺寸，与输入尺寸无关（自动推算步长/填充）
adaptive_pool = nn.AdaptiveAvgPool3d(output_size=(1, 1, 1))  # 全局平均池化：输出 (N, C, 1, 1, 1)

with torch.no_grad():  # 关闭梯度计算（仅演示）
    out_max  = max_pool(feat_3d)       # MaxPool3d 输出
    out_avg  = avg_pool(feat_3d)       # AvgPool3d 输出
    out_adap = adaptive_pool(feat_3d)  # AdaptiveAvgPool3d 输出

print("3D 池化输出形状：")
print(f"  输入特征图:             {feat_3d.shape}")
print(f"  MaxPool3d(kernel=2,s=2):  {out_max.shape}")
print(f"  AvgPool3d(kernel=2,s=2):  {out_avg.shape}")
print(f"  AdaptiveAvgPool3d(1,1,1): {out_adap.shape}")
print()
# 自适应全局池化后 squeeze 掉空间维度，得到 1D 特征向量，用于分类
feat_vec = out_adap.squeeze(-1).squeeze(-1).squeeze(-1)  # (N, C, 1, 1, 1) → (N, C)
print(f"  展平后特征向量形状:        {feat_vec.shape}  ← 可直接接全连接层")

## 五、3D 转置卷积上采样（ConvTranspose3d）

`nn.ConvTranspose3d` 是 `nn.Conv3d` 的转置（逆）操作，用于 **3D 特征图的上采样**，是 3D U-Net 解码器、3D GAN 生成器等的核心组件。

**输出尺寸公式**（与 Conv3d 相反）：

$$
D_{\text{out}} = (D_{\text{in}} - 1) \times s_D - 2p_D + k_D
$$

**避免棋盘格伪影的原则**（与 2D 相同）：`kernel_size % stride == 0`

In [ ]:
torch.manual_seed(seed)  # 固定权重初始化

# ── ConvTranspose3d：3D 转置卷积上采样层 ─────────────────────────────────────
deconv3d = nn.ConvTranspose3d(
    in_channels=64,   # 输入通道数
    out_channels=32,  # 输出通道数（通道减半，常见设计）
    kernel_size=2,    # 卷积核 2×2×2（能被 stride=2 整除，无棋盘格伪影）
    stride=2,         # 步长 2：每个维度精确上采样 2 倍
    bias=False        # 不使用偏置
)  # 权重形状 (in_channels, out_channels, kD, kH, kW) = (64, 32, 2, 2, 2)

# 模拟编码器瓶颈层输出：低分辨率深层特征
bottleneck = torch.randn(1, 64, 4, 16, 16)  # 形状 (N=1, C=64, D=4, H=16, W=16)

with torch.no_grad():  # 关闭梯度
    up3d_out = deconv3d(bottleneck)  # 前向传播：(1,64,4,16,16) → (1,32,8,32,32)

print("ConvTranspose3d 上采样：")
print(f"  输入（瓶颈特征）: {bottleneck.shape}")
print(f"  输出（上采样后）: {up3d_out.shape}")
print(f"  尺寸变化: D {bottleneck.shape[2]}→{up3d_out.shape[2]},",
      f"H {bottleneck.shape[3]}→{up3d_out.shape[3]},",
      f"W {bottleneck.shape[4]}→{up3d_out.shape[4]}")
print()
# 参数量展示
params_deconv = sum(p.numel() for p in deconv3d.parameters())  # 总参数量（int）
print(f"  ConvTranspose3d 参数量: {params_deconv:,}  = 64×32×2×2×2")

## 六、3D 卷积网络实战：简化版 C3D 视频分类

### 6.1 C3D 网络简介

**C3D（3D Convolutional Networks for Video Classification）** 是 Du Tran et al. 2015 年提出的经典视频理解网络，首次系统证明 3D 卷积在视频动作识别上优于 2D 卷积。

**核心设计**：
- 所有卷积核均为 `3×3×3`（时间和空间感受野对称）
- 输入为 16 帧 × 112×112 的 RGB 视频片段
- 通过多层 `Conv3d + MaxPool3d` 提取时空特征，最后接全连接层分类

In [ ]:
class SimplifiedC3D(nn.Module):
    """简化版 C3D 视频分类网络（用于教学演示）。

    原始 C3D 输入为 (N, 3, 16, 112, 112)；此简化版使用 (N, 3, 8, 32, 32)
    缩减输入规模以降低内存需求，核心结构与原版一致。

    网络结构：
        Conv3d(3→32, k=3, p=1)  → BN → ReLU
        MaxPool3d(2)             → [N, 32, 4, 16, 16]
        Conv3d(32→64, k=3, p=1) → BN → ReLU
        MaxPool3d(2)             → [N, 64, 2,  8,  8]
        Conv3d(64→128, k=3, p=1)→ BN → ReLU
        AdaptiveAvgPool3d(1,1,1) → [N, 128]
        Linear(128 → num_classes)

    参数:
        num_classes (int): 分类类别数，默认 10
    """
    def __init__(self, num_classes=10):
        super().__init__()  # 调用父类 nn.Module 初始化方法

        # ── 第一卷积块：3→32 通道，时空 same padding ────────────────────────
        self.block1 = nn.Sequential(  # 顺序容器，依次执行子模块
            nn.Conv3d(3, 32, kernel_size=3, padding=1, bias=False),  # 3D 卷积，same padding
            nn.BatchNorm3d(32),   # 3D 批归一化：对 (N, D, H, W) 四个维度的统计量做归一化
            nn.ReLU(inplace=True), # ReLU 激活，inplace 节省显存
            nn.MaxPool3d(kernel_size=2, stride=2),  # 3D 最大池化：D/H/W 各缩小 2 倍
        )  # 输出形状：(N, 32, 4, 16, 16)（输入为 (N,3,8,32,32)）

        # ── 第二卷积块：32→64 通道 ────────────────────────────────────────────
        self.block2 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, padding=1, bias=False),  # 3D 卷积，same padding
            nn.BatchNorm3d(64),    # 3D 批归一化
            nn.ReLU(inplace=True), # ReLU 激活
            nn.MaxPool3d(kernel_size=2, stride=2),  # 3D 最大池化
        )  # 输出形状：(N, 64, 2, 8, 8)

        # ── 第三卷积块：64→128 通道 ───────────────────────────────────────────
        self.block3 = nn.Sequential(
            nn.Conv3d(64, 128, kernel_size=3, padding=1, bias=False),  # 3D 卷积
            nn.BatchNorm3d(128),   # 3D 批归一化
            nn.ReLU(inplace=True), # ReLU 激活
        )  # 输出形状：(N, 128, 2, 8, 8)

        # ── 全局自适应平均池化：将任意时空尺寸压缩为 1×1×1 ──────────────────
        self.global_pool = nn.AdaptiveAvgPool3d(output_size=(1, 1, 1))  # 全局池化，输出 (N,128,1,1,1)

        # ── 分类头：128 维特征向量 → num_classes 类别分数 ────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),              # 展平：(N, 128, 1, 1, 1) → (N, 128)；Flatten 默认展平 dim=1 起的所有维度
            nn.Linear(128, 64),        # 全连接层：128 → 64，权重形状 (64, 128)
            nn.ReLU(inplace=True),     # ReLU 激活
            nn.Dropout(p=0.5),         # Dropout：训练时随机置零 50% 神经元，防止过拟合
            nn.Linear(64, num_classes), # 输出层：64 → num_classes，权重形状 (num_classes, 64)
        )  # 输出形状：(N, num_classes)

    def forward(self, x):
        """前向传播：Block1 → Block2 → Block3 → GlobalPool → Classifier。

        参数:
            x (torch.Tensor): 输入视频片段，形状 (N, 3, D, H, W)

        返回:
            torch.Tensor: 分类 logits，形状 (N, num_classes)
        """
        x = self.block1(x)       # 时空卷积 + 池化：(N,3,D,H,W) → (N,32,D/2,H/2,W/2)
        x = self.block2(x)       # 时空卷积 + 池化：(N,32,...) → (N,64,D/4,H/4,W/4)
        x = self.block3(x)       # 时空卷积（无池化）：(N,64,...) → (N,128,...)
        x = self.global_pool(x)  # 全局平均池化：(N,128,*,*,*) → (N,128,1,1,1)
        x = self.classifier(x)   # 展平 + 全连接分类：(N,128,1,1,1) → (N,num_classes)
        return x                 # 返回形状 (N, num_classes) 的 logits 张量


torch.manual_seed(seed)   # 固定随机种子
c3d_model = SimplifiedC3D(num_classes=10)  # 实例化 C3D 模型，10 分类

# 统计总参数量
total_params = sum(p.numel() for p in c3d_model.parameters())      # 全部参数总数（int）
trainable_params = sum(p.numel() for p in c3d_model.parameters() if p.requires_grad)  # 可训练参数数

print("SimplifiedC3D 模型参数统计：")
print(f"  总参数量: {total_params:,}")
print(f"  可训练参数量: {trainable_params:,}")
print()

In [ ]:
# ── 前向传播各层形状追踪 ──────────────────────────────────────────────────────

torch.manual_seed(seed)  # 固定随机种子

# 模拟一个 batch 的视频输入（N=2，3 通道，8 帧，32×32）
video_batch = torch.randn(2, 3, 8, 32, 32)  # 形状 (N=2, C=3, D=8, H=32, W=32)

c3d_model.eval()  # 切换到评估模式：关闭 Dropout（推理阶段）
with torch.no_grad():  # 关闭梯度计算
    # 逐层追踪形状变化
    x0 = video_batch               # 原始输入
    x1 = c3d_model.block1(x0)     # Block1 输出
    x2 = c3d_model.block2(x1)     # Block2 输出
    x3 = c3d_model.block3(x2)     # Block3 输出
    x4 = c3d_model.global_pool(x3)# 全局池化输出
    x5 = c3d_model.classifier(x4) # 分类头输出（logits）

print("SimplifiedC3D 各层输出形状追踪（batch=2, 8帧, 32×32）：")
print(f"  输入:         {x0.shape}")
print(f"  Block1 输出:  {x1.shape}")
print(f"  Block2 输出:  {x2.shape}")
print(f"  Block3 输出:  {x3.shape}")
print(f"  全局池化:     {x4.shape}")
print(f"  分类输出:     {x5.shape}  ← logits，对应 10 个类别")

# 可视化模型各层通道数变化
layer_names   = ['输入', 'Block1\n(Conv+Pool)', 'Block2\n(Conv+Pool)', 'Block3\n(Conv)', '全局池化', '分类输出']  # 层名
channel_sizes = [3, 32, 64, 128, 128, 10]  # 各层的通道（或类别）数
spatial_sizes = [                           # 各层的空间尺寸字符串（D×H×W）
    '8×32×32', '4×16×16', '2×8×8', '2×8×8', '1×1×1', '-'
]

fig, ax = plt.subplots(figsize=(12, 4))  # 创建画布
colors_bar = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']  # 每层条形颜色
bars = ax.bar(layer_names, channel_sizes, color=colors_bar, alpha=0.85, width=0.5)  # 绘制条形图

# 在每个条形顶部标注通道数和空间尺寸
for bar, ch, sp in zip(bars, channel_sizes, spatial_sizes):  # 同时遍历条形、通道数、空间尺寸
    ax.text(bar.get_x() + bar.get_width() / 2,  # x 中心
            bar.get_height() + 2,               # y：条形顶部上方
            f'C={ch}\n{sp}',                    # 标注内容：通道数 + 空间尺寸
            ha='center', va='bottom', fontsize=9)  # 居中对齐

ax.set_ylabel('通道数 / 类别数', fontsize=11)              # y 轴标签
ax.set_title('SimplifiedC3D 各层通道数变化', fontsize=13)  # 图标题
ax.spines['top'].set_visible(False)    # 隐藏顶部边框
ax.spines['right'].set_visible(False)  # 隐藏右侧边框
plt.tight_layout()  # 自动调整间距
plt.show()          # 显示图表

## 七、医学影像应用：3D U-Net 编解码器

### 7.1 3D U-Net 简介

**3D U-Net**（Çiçek et al. 2016）是原始 2D U-Net 的三维扩展，专为 **CT / MRI 体积数据**的像素级（体素级）分割而设计。

**输入**：`(N, 1, D, H, W)` 的灰度体数据（CT 值 / MRI 强度）  
**输出**：`(N, num_classes, D, H, W)` 的逐体素分割掩码

**关键差异（vs 2D U-Net）**：
- 所有卷积替换为 `Conv3d`，池化替换为 `MaxPool3d`，转置卷积替换为 `ConvTranspose3d`
- Skip Connection 在三维空间拼接，融合更丰富的立体上下文信息

In [ ]:
class DoubleConv3d(nn.Module):
    """3D U-Net 基础双层卷积块：Conv3d → BN → ReLU → Conv3d → BN → ReLU。

    参数:
        in_channels (int): 输入通道数
        out_channels (int): 输出通道数
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()  # 初始化父类
        self.double_conv = nn.Sequential(  # 顺序容器
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),  # 第一个 3×3×3 卷积
            nn.BatchNorm3d(out_channels),   # 批归一化（3D）
            nn.ReLU(inplace=True),          # ReLU 激活
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),  # 第二个 3×3×3 卷积
            nn.BatchNorm3d(out_channels),   # 批归一化（3D）
            nn.ReLU(inplace=True),          # ReLU 激活
        )  # 整体保持空间尺寸不变（same padding）

    def forward(self, x):
        """前向传播：双层卷积块。

        参数:
            x (torch.Tensor): 输入特征图，形状 (N, in_channels, D, H, W)

        返回:
            torch.Tensor: 卷积后特征图，形状 (N, out_channels, D, H, W)
        """
        return self.double_conv(x)  # 顺序执行两个 Conv3d-BN-ReLU 块


class UNet3D(nn.Module):
    """简化版 3D U-Net，用于医学体积数据分割（教学演示）。

    编码器：2 个 DoubleConv3d + MaxPool3d 下采样
    瓶颈层：1 个 DoubleConv3d
    解码器：2 个 ConvTranspose3d 上采样 + Skip Concat + DoubleConv3d

    参数:
        in_channels (int): 输入通道数，CT 灰度图为 1
        num_classes (int): 分割类别数（体素级输出通道数）
        base_ch (int): 基础通道数，默认 16
    """
    def __init__(self, in_channels=1, num_classes=2, base_ch=16):
        super().__init__()  # 初始化父类

        # ── 编码器（Encoder / Contracting Path）──────────────────────────────
        self.enc1 = DoubleConv3d(in_channels, base_ch)       # 层1：in → base_ch（16）
        self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)   # 下采样 2×：D/H/W 减半
        self.enc2 = DoubleConv3d(base_ch, base_ch * 2)       # 层2：base_ch → 2×base_ch（32）
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)   # 下采样 2×

        # ── 瓶颈层（Bottleneck）─────────────────────────────────────────────
        self.bottleneck = DoubleConv3d(base_ch * 2, base_ch * 4)  # 32 → 64 通道（最深层语义）

        # ── 解码器（Decoder / Expanding Path）───────────────────────────────
        # ConvTranspose3d(kernel=2,stride=2)：精确 2× 上采样，无棋盘格伪影
        self.up2 = nn.ConvTranspose3d(base_ch * 4, base_ch * 2, kernel_size=2, stride=2)  # 上采样 2×，64→32
        self.dec2 = DoubleConv3d(base_ch * 4, base_ch * 2)  # Skip Concat 后：32+32=64 → 32

        self.up1 = nn.ConvTranspose3d(base_ch * 2, base_ch, kernel_size=2, stride=2)  # 上采样 2×，32→16
        self.dec1 = DoubleConv3d(base_ch * 2, base_ch)      # Skip Concat 后：16+16=32 → 16

        # ── 输出层：1×1×1 卷积将通道数映射到分割类别数 ─────────────────────
        self.out_conv = nn.Conv3d(base_ch, num_classes, kernel_size=1)  # 1×1×1 卷积，不改变空间尺寸

    def forward(self, x):
        """前向传播：编码 → 瓶颈 → 解码（含 Skip Connection）→ 输出。

        参数:
            x (torch.Tensor): CT/MRI 体积输入，形状 (N, in_channels, D, H, W)

        返回:
            torch.Tensor: 分割 logits，形状 (N, num_classes, D, H, W)
        """
        # 编码路径（保存 Skip Connection 特征）
        e1 = self.enc1(x)       # 编码层1：(N,in_c,D,H,W) → (N,base_ch,D,H,W)
        e2 = self.enc2(self.pool1(e1))  # 下采样+编码层2：(N,base_ch,...) → (N,base_ch×2,D/2,H/2,W/2)

        # 瓶颈层
        b = self.bottleneck(self.pool2(e2))  # 下采样+瓶颈：(N,base_ch×2,...) → (N,base_ch×4,D/4,H/4,W/4)

        # 解码路径（融合 Skip Connection）
        d2 = self.up2(b)                     # 转置卷积上采样：(N,base_ch×4,...) → (N,base_ch×2,D/2,H/2,W/2)
        d2 = torch.cat([d2, e2], dim=1)      # 通道拼接 Skip：(N,base_ch×4,D/2,H/2,W/2)（32+32=64）
        d2 = self.dec2(d2)                   # 双层卷积融合：64 → 32

        d1 = self.up1(d2)                    # 转置卷积上采样：(N,base_ch×2,...) → (N,base_ch,D,H,W)
        d1 = torch.cat([d1, e1], dim=1)      # 通道拼接 Skip：(N,base_ch×2,D,H,W)（16+16=32）
        d1 = self.dec1(d1)                   # 双层卷积融合：32 → 16

        out = self.out_conv(d1)              # 1×1×1 卷积：16 → num_classes
        return out                           # 返回形状 (N, num_classes, D, H, W) 的分割 logits


torch.manual_seed(seed)  # 固定种子
unet3d = UNet3D(in_channels=1, num_classes=2, base_ch=16)  # CT 灰度输入，二分类（前景/背景）

# 统计参数量
unet3d_params = sum(p.numel() for p in unet3d.parameters() if p.requires_grad)  # 可训练参数总数
print(f"3D U-Net 可训练参数量: {unet3d_params:,}")
print()

In [ ]:
# ── 3D U-Net 前向传播验证与形状追踪 ────────────────────────────────────────────

# 模拟一个 CT 体数据批次：batch=1，单通道，32×64×64
ct_input = torch.randn(1, 1, 32, 64, 64)  # 形状 (N=1, C=1, D=32切片, H=64, W=64)

unet3d.eval()  # 切换评估模式，关闭 Dropout（若有）
with torch.no_grad():  # 关闭梯度
    # 手动逐步追踪形状
    e1_out  = unet3d.enc1(ct_input)                        # 编码层1 输出
    e2_out  = unet3d.enc2(unet3d.pool1(e1_out))            # 池化→编码层2 输出
    b_out   = unet3d.bottleneck(unet3d.pool2(e2_out))      # 池化→瓶颈输出
    d2_up   = unet3d.up2(b_out)                            # 解码上采样2
    d2_out  = unet3d.dec2(torch.cat([d2_up, e2_out], 1))  # Skip Concat + 解码卷积2
    d1_up   = unet3d.up1(d2_out)                           # 解码上采样1
    d1_out  = unet3d.dec1(torch.cat([d1_up, e1_out], 1))  # Skip Concat + 解码卷积1
    seg_out = unet3d.out_conv(d1_out)                      # 分割输出

print("3D U-Net 各层形状追踪（输入: 1×1×32×64×64）：")
print()
print("  ── 编码路径（Encoder）────────────────────────")
print(f"  输入 CT 体数据:     {ct_input.shape}")
print(f"  Enc1 输出:          {e1_out.shape}")
print(f"  Pool1 + Enc2 输出:  {e2_out.shape}")
print(f"  Pool2 + Bottleneck: {b_out.shape}")
print()
print("  ── 解码路径（Decoder）────────────────────────")
print(f"  Up2（转置卷积）:    {d2_up.shape}")
print(f"  Dec2（Skip+Conv）:  {d2_out.shape}")
print(f"  Up1（转置卷积）:    {d1_up.shape}")
print(f"  Dec1（Skip+Conv）:  {d1_out.shape}")
print(f"  分割输出:           {seg_out.shape}  ← 与输入等尺寸")
print()
print(f"  输出尺寸是否与输入完全一致: {ct_input.shape[2:] == seg_out.shape[2:]}")

In [ ]:
# ── 可视化 3D U-Net 的编解码器特征图切片 ────────────────────────────────────

# 选取中间切片（D 维的第 16 帧）展示各层特征的空间分辨率变化
mid_d = 16  # 选取深度维度的中间切片索引

fig, axes = plt.subplots(2, 4, figsize=(16, 7))  # 创建 2×4 子图布局

# ── 第一行：编码路径各层的第 0 通道特征图 ──────────────────────────────────
layers_enc = [  # 编码路径各层数据与标题的列表
    (ct_input[0, 0].numpy(),          '输入 CT (D=32, H=64, W=64)'),  # 原始输入第 0 通道
    (e1_out[0, 0].numpy(),            'Enc1 (16ch, D=32, H=64, W=64)'),  # 编码层1
    (e2_out[0, 0].numpy(),            'Enc2 (32ch, D=16, H=32, W=32)'),  # 编码层2
    (b_out[0, 0].numpy(),             'Bottleneck (64ch, D=8, H=16, W=16)'),  # 瓶颈层
]  # 列表中每个元素为 (3D numpy 数组, 标题字符串)

for col, (vol, title) in enumerate(layers_enc):  # 遍历编码各层（4 个）
    # 取中间切片：若体积深度 < mid_d 则取中间帧
    d_idx = min(mid_d, vol.shape[0] - 1)  # 防止索引越界
    slice_2d = vol[d_idx]                  # 取第 d_idx 深度切片，形状 (H, W)
    axes[0, col].imshow(slice_2d, cmap='viridis')  # 显示为热力图
    axes[0, col].set_title(title, fontsize=8.5)    # 子图标题
    axes[0, col].axis('off')                       # 关闭坐标轴

# ── 第二行：解码路径各层的第 0 通道特征图 ──────────────────────────────────
layers_dec = [  # 解码路径各层数据与标题的列表
    (d2_up[0, 0].numpy(),             'Up2 后 (32ch, D=16, H=32, W=32)'),   # 上采样2
    (d2_out[0, 0].numpy(),            'Dec2 后 (32ch, D=16, H=32, W=32)'),  # 解码层2
    (d1_out[0, 0].numpy(),            'Dec1 后 (16ch, D=32, H=64, W=64)'),  # 解码层1
    (seg_out[0, 0].detach().numpy(),  '分割输出 (2ch, D=32, H=64, W=64)'),  # 分割结果
]

for col, (vol, title) in enumerate(layers_dec):  # 遍历解码各层（4 个）
    d_idx = min(mid_d, vol.shape[0] - 1)  # 防止索引越界
    slice_2d = vol[d_idx]                  # 取中间切片，形状 (H, W)
    axes[1, col].imshow(slice_2d, cmap='viridis')  # 显示为热力图
    axes[1, col].set_title(title, fontsize=8.5)    # 子图标题
    axes[1, col].axis('off')                       # 关闭坐标轴

plt.suptitle('3D U-Net 编解码路径特征图中间切片可视化', fontsize=13)  # 整体标题
plt.tight_layout()  # 自动调整间距
plt.show()          # 渲染显示

## 八、3D 卷积的高效变体对比

标准 3D 卷积的计算量和参数量是 2D 卷积的 `kernel_d` 倍，在工程中代价较高。研究者提出了几种高效变体：

### 8.1 三种变体原理

| 变体 | 核心思路 | 卷积核形状 | 参数量（相对标准 3D） |
|------|---------|-----------|---------------------|
| **标准 3D 卷积** | 时空联合建模 | `(kD, kH, kW)` | 1× |
| **(2+1)D 卷积** | 分解为空间 2D + 时间 1D 两步 | `(1, kH, kW)` + `(kD, 1, 1)` | ≈ 0.6×，性能相近 |
| **Pseudo-3D（P3D）** | 空间 2D 后接时间 1D（残差形式） | `(1, kH, kW)` + `(kD, 1, 1)` | ≈ 0.7× |

### 8.2 (2+1)D 卷积实现

In [ ]:
class Conv2Plus1D(nn.Module):
    """(2+1)D 卷积块：将 3D 卷积分解为空间 2D 卷积 + 时间 1D 卷积。

    理论依据：R(2+1)D 论文（Tran et al., 2018）
    - 先用 (1×kH×kW) 做空间建模（跨帧无交互）
    - 再用 (kD×1×1) 做时间建模（跨帧交互）
    - 分解后引入了额外的非线性（两层 ReLU），表达能力不弱于整体 3D 卷积

    参数:
        in_channels (int): 输入通道数
        out_channels (int): 输出通道数
        spatial_k (int): 空间卷积核大小（H/W 方向），默认 3
        temporal_k (int): 时间卷积核大小（D 方向），默认 3
    """
    def __init__(self, in_channels, out_channels, spatial_k=3, temporal_k=3):
        super().__init__()  # 初始化父类

        # 中间通道数：用于连接空间卷积和时间卷积
        # 原论文设计公式保证参数量与标准 3D 卷积近似
        mid_channels = int(
            (in_channels * out_channels * spatial_k**2 * temporal_k) /
            (in_channels * spatial_k**2 + out_channels * temporal_k)
        )  # 中间通道数（int）

        # 空间 2D 卷积：kernel=(1, kH, kW)，只在 H/W 方向滑动，不跨帧
        self.spatial_conv = nn.Sequential(
            nn.Conv3d(
                in_channels, mid_channels,
                kernel_size=(1, spatial_k, spatial_k),  # D 方向 kernel=1：不跨帧
                padding=(0, spatial_k//2, spatial_k//2),# H/W same padding；D 不填充
                bias=False
            ),  # 空间卷积，权重形状 (mid_ch, in_ch, 1, kH, kW)
            nn.BatchNorm3d(mid_channels),  # 批归一化
            nn.ReLU(inplace=True),         # 激活（分解引入额外非线性）
        )

        # 时间 1D 卷积：kernel=(kD, 1, 1)，只在 D（时间）方向滑动，不跨像素
        self.temporal_conv = nn.Sequential(
            nn.Conv3d(
                mid_channels, out_channels,
                kernel_size=(temporal_k, 1, 1),  # H/W 方向 kernel=1：不跨空间
                padding=(temporal_k//2, 0, 0),   # D same padding；H/W 不填充
                bias=False
            ),  # 时间卷积，权重形状 (out_ch, mid_ch, kD, 1, 1)
            nn.BatchNorm3d(out_channels),  # 批归一化
            nn.ReLU(inplace=True),         # 激活
        )

    def forward(self, x):
        """前向传播：空间 2D 卷积 → 时间 1D 卷积。

        参数:
            x (torch.Tensor): 输入视频特征，形状 (N, in_channels, D, H, W)

        返回:
            torch.Tensor: 时空特征，形状 (N, out_channels, D, H, W)（same padding）
        """
        x = self.spatial_conv(x)   # 空间建模：跨像素，不跨帧
        x = self.temporal_conv(x)  # 时间建模：跨帧，不跨像素
        return x                   # 返回时空联合特征


torch.manual_seed(seed)  # 固定初始化

# 对比标准 Conv3d 和 (2+1)D Conv 的参数量与前向时间
in_ch, out_ch = 64, 128  # 输入/输出通道数
std_conv3d = nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)  # 标准 3D 卷积
decomp_conv = Conv2Plus1D(in_ch, out_ch, spatial_k=3, temporal_k=3)          # (2+1)D 分解卷积

params_std   = sum(p.numel() for p in std_conv3d.parameters())   # 标准 Conv3d 参数量（int）
params_decomp = sum(p.numel() for p in decomp_conv.parameters()) # (2+1)D 参数量（int）

print(f"标准 Conv3d（in=64, out=128, k=3）参数量: {params_std:,}")
print(f"(2+1)D 分解卷积参数量:                   {params_decomp:,}")
print(f"参数量比例:                               {params_decomp/params_std:.2f}×")
print()

# 前向传播形状验证
feat = torch.randn(1, 64, 8, 32, 32)  # 模拟输入特征图
with torch.no_grad():  # 关闭梯度
    out_std   = std_conv3d(feat)    # 标准 Conv3d 输出
    out_decomp = decomp_conv(feat)  # (2+1)D 输出

print(f"标准 Conv3d 输出形状:   {out_std.shape}")
print(f"(2+1)D Conv 输出形状:   {out_decomp.shape}")

### 8.3 各方法综合对比

In [ ]:
import pandas as pd  # 导入 pandas，用于创建总结表格

# 构建各 3D 卷积变体综合对比表
comparison_data = {
    '方法': [
        '标准 2D 卷积\n(Conv2d)',
        '标准 3D 卷积\n(Conv3d)',
        '(2+1)D 卷积\n(R(2+1)D)',
        '伪 3D 卷积\n(P3D / Pseudo-3D)',
        '深度可分离 3D\n(Depthwise Conv3d)',
    ],
    '时序建模': ['❌ 无', '✅ 联合', '✅ 分解', '✅ 分解', '✅ 联合（轻量）'],
    '参数量': ['最少', '最多', '中等（≈0.6×3D）', '中等（≈0.7×3D）', '极少（≈1/C×3D）'],
    '计算量': ['最少', '最多', '较少', '较少', '极少'],
    '优点': [
        '计算快、适合单帧任务',
        '时空联合建模最强',
        '引入额外非线性、精度近似3D',
        '残差结构更易训练',
        '移动端/边缘设备部署',
    ],
    '典型应用': [
        '图像分类/检测',
        'C3D、3D U-Net、CT 分割',
        'R(2+1)D 视频动作识别',
        'P3D-ResNet 视频理解',
        '轻量级视频模型',
    ],
}  # 字典：键为列名，值为各方法的属性列表

df_cmp = pd.DataFrame(comparison_data)  # 创建 DataFrame，形状 (5, 6)

pd.set_option('display.max_colwidth', None)   # 不限制列宽
pd.set_option('display.max_columns', None)    # 显示所有列
pd.set_option('display.width', 200)           # 设置显示宽度

print("=" * 100)
print("3D 卷积各变体综合对比")
print("=" * 100)
print(df_cmp.to_string(index=False))  # 打印完整 DataFrame（不显示行索引）
print("=" * 100)

In [ ]:
# ── 参数量对比雷达图 ─────────────────────────────────────────────────────────

# 以 in=64, out=128, kernel=3 为基准，统计各方法实际参数量
torch.manual_seed(seed)  # 固定初始化

# 深度可分离 3D 卷积：逐通道 3D 卷积 + 1×1×1 逐点卷积
depthwise_conv3d_params = (
    in_ch * 3**3 +      # 深度卷积：每通道独立一个 3×3×3 核，groups=in_ch
    in_ch * out_ch      # 逐点卷积：1×1×1，将 in_ch 通道映射到 out_ch
)  # 近似参数量（int）

# 各方法参数量（in=64, out=128, k=3）
method_names = [
    'Conv2d\n(2D 基准)',
    'Conv3d\n(标准 3D)',
    '(2+1)D Conv',
    '深度可分离\nConv3d',
]  # 方法名称列表（str）

param_counts = [
    in_ch * out_ch * 3**2,        # Conv2d：64×128×3×3 = 73728
    params_std,                    # Conv3d：已计算
    params_decomp,                 # (2+1)D：已计算
    depthwise_conv3d_params,       # 深度可分离：近似值
]  # 各方法参数量列表（int）

bar_colors = ['#4C72B0', '#C44E52', '#55A868', '#8172B2']  # 条形颜色

fig, ax = plt.subplots(figsize=(10, 5))  # 创建画布
bars = ax.bar(method_names, param_counts, color=bar_colors, alpha=0.85, width=0.5)  # 绘制条形图

# 在每个条形顶部标注参数量（千为单位）
for bar, cnt in zip(bars, param_counts):  # 遍历条形与参数量
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # x 中心
        bar.get_height() + 500,              # y：条形顶部上方
        f'{cnt/1000:.1f}K',                  # 显示参数量（K = 千）
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )  # 参数量标注

ax.set_ylabel('参数量', fontsize=12)                               # y 轴标签
ax.set_title('各卷积方法参数量对比（in=64, out=128, kernel=3）', fontsize=13)  # 图标题
ax.spines['top'].set_visible(False)    # 隐藏顶部边框
ax.spines['right'].set_visible(False)  # 隐藏右侧边框
ax.yaxis.set_major_formatter(           # y 轴刻度格式：显示千位数
    plt.FuncFormatter(lambda x, _: f'{int(x/1000)}K')
)  # FuncFormatter：自定义刻度格式化函数

plt.tight_layout()  # 自动调整间距
plt.show()          # 显示图表

## 九、总结

### 9.1 3D 卷积核心知识速查

| 知识点 | 关键内容 |
|--------|--------|
| **输入张量形状** | `(N, C, D, H, W)`，比 2D 多一个深度/时间维 D |
| **输出尺寸公式** | $D_{out} = \lfloor(D_{in} + 2p - d(k-1) - 1)/s + 1\rfloor$（H/W 同理） |
| **Same Padding** | `padding = kernel_size // 2`（stride=1, dilation=1 时） |
| **权重形状** | `(C_out, C_in, kD, kH, kW)` |
| **参数量** | `C_out × C_in × kD × kH × kW`（是同配置 2D 卷积的 kD 倍） |
| **3D 池化** | `MaxPool3d`、`AvgPool3d`、`AdaptiveAvgPool3d` |
| **3D 上采样** | `ConvTranspose3d`（kernel=2,stride=2 无棋盘格） |

### 9.2 各场景选用指南

```
你的数据有时序 / 体积结构？
│
├─ 视频动作识别？
│   ├─ 追求精度（算力充足）→ 标准 Conv3d（C3D、SlowFast）
│   └─ 追求效率（部署优先）→ (2+1)D Conv（R(2+1)D）或 Pseudo-3D
│
├─ 医学影像分割（CT/MRI）？
│   └─ 3D U-Net（Conv3d + ConvTranspose3d + Skip Connection）
│
├─ 移动端 / 边缘设备？
│   └─ 深度可分离 Conv3d（极大降低参数量与计算量）
│
└─ 数据量少 / 计算受限？
      └─ 用预训练 2D 模型逐帧提取特征 + LSTM/Transformer 时序建模
           （避免从零训练 3D 网络，节省数据和算力）
```

### 9.3 关键结论

1. **3D 卷积 = 2D 卷积在深度/时间维度的扩展**：输入多一个 D 维，权重多一个 kD 维，参数量是 2D 的 kD 倍
2. **计算代价显著**：内存和 FLOPs 均随 D 维线性增长，工程中需谨慎权衡精度与效率
3. **(2+1)D 分解是高效替代**：将时空联合卷积分解为空间+时间两步，参数量减少约 40%，精度接近甚至超过标准 3D 卷积
4. **医学影像首选 3D U-Net**：CT/MRI 体数据的立体上下文信息至关重要，3D 卷积能捕获 2D 逐帧处理无法获取的跨切片依赖
5. **ConvTranspose3d 的 kernel 必须整除 stride**：与 2D 相同，`kernel=2, stride=2` 是避免三维棋盘格伪影的工程最佳实践